# IGV notebook demo

This notebook uses `igv-notebook` to embed igv.js in a VS Code Jupyter notebook. It loads a local BAM with a BAI index and a reference FASTA with a FAI index.

## Requirements

- `pip install igv-notebook` in the active kernel environment
- Ensure the `.bai` and `.fai` index files exist alongside the BAM and FASTA files

In [ ]:
import igv_notebook

igv_notebook.init()

In [ ]:
# Paths from run.sh
BAM_FILE = "/home/peterkad/pkadmaster/data/mutationalscanning_bam/ph/diploid_assembly/ph_plus_unmapped_diploid_v2.bam"
FASTA_FILE = "/home/peterkad/pkadmaster/data/ph/ph_diploid.fa"

In [ ]:
import os
from pathlib import Path

BAM_INDEX = BAM_FILE + ".bai"
FASTA_INDEX = FASTA_FILE + ".fai"

for label, path in {
    "BAM": BAM_FILE,
    "BAM_INDEX": BAM_INDEX,
    "FASTA": FASTA_FILE,
    "FASTA_INDEX": FASTA_INDEX,
}.items():
    p = Path(path)
    exists = p.exists()
    size = p.stat().st_size if exists else None
    readable = os.access(p, os.R_OK) if exists else False
    print(f"{label}: exists={exists}, readable={readable}, size={size}, path={p}")

# Try opening a small chunk to confirm read access
with open(BAM_FILE, "rb") as bam_handle:
    bam_handle.read(64)
with open(FASTA_FILE, "rb") as fasta_handle:
    fasta_handle.read(64)

print("Basic read checks passed.")

In [ ]:
# Pick a contig from the FASTA index so the locus exists
with open(FASTA_INDEX, "r", encoding="utf-8") as fai_handle:
    first_line = fai_handle.readline().strip()

if not first_line:
    raise ValueError("FASTA index is empty.")

contig_name = first_line.split("\t", 1)[0]
locus = f"{contig_name}:1-1000"
print(f"Using locus: {locus}")

In [ ]:
from IPython.display import display

igv_browser = igv_notebook.Browser(
    {
        "genome": "custom",
        "reference": {
            "id": "custom",
            "name": "custom",
            "fastaPath": FASTA_FILE,
            "indexPath": FASTA_INDEX,
        },
        "locus": locus,
    }
)

display(igv_browser)

In [ ]:
# Load the BAM track after the browser is displayed
igv_browser.load_track(
    {
        "name": "BAM",
        "path": BAM_FILE,
        "indexPath": BAM_INDEX,
        "format": "bam",
        "type": "alignment",
    }
)

# Navigate to a known contig region
igv_browser.search(locus)